# 부산광역시 생활인구 비율과 인구밀집구역(DID) 분석

행정동별 최신 생활인구 비율 지도와 2015·2024년 인구밀집구역(DID) 분석을 각각 편집할 수 있는 셀로 구성했습니다. 지도는 CARTO Positron 배경을 사용하고, 지도는 노트북 셀 출력으로 표시합니다.

- 생활인구 비율: 2026년 9월 생활인구 / 주민등록인구
- DID: 2015년·2024년 500m 격자 자료
- 부산 전체 격자를 회색으로 표시하고 DID 상태(유지·탈락·신규)만 색상으로 강조합니다. 회색 격자 범례는 표시하지 않습니다.
- 배경지도는 공개 CARTO Positron 타일을 사용합니다.


In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import folium
from dotenv import load_dotenv
from IPython.display import IFrame, display

# 노트북을 저장소 폴더 또는 상위 작업 폴더에서 실행해도 data/를 찾습니다.
cwd = Path.cwd()
if (cwd / "data" / "busan_living_population_202609.csv").exists():
    project_dir = cwd
elif (cwd / "shrinking-busan" / "data" / "busan_living_population_202609.csv").exists():
    project_dir = cwd / "shrinking-busan"
else:
    raise FileNotFoundError("shrinking-busan/data 파일을 찾을 수 없습니다.")
data_dir = project_dir / "data"

# 현재 폴더 또는 상위 폴더의 .env를 읽습니다. CARTO 기본 타일은 별도 키 없이 사용할 수 있습니다.
for env_dir in (cwd, *cwd.parents):
    env_path = env_dir / ".env"
    if env_path.exists():
        load_dotenv(env_path, override=False)
        break

# 2026년 9월 행정동별 시간대 생활인구와 월말 주민등록인구
living = pd.read_csv(
    data_dir / "busan_living_population_202609.csv",
    dtype={"행정동코드": "string"},
)
resident = pd.read_csv(
    data_dir / "busan_resident_population_202609.csv",
    dtype={"행정기관코드": "string"},
)
living["행정동코드"] = living["행정동코드"].str.strip().str.zfill(10)
resident["행정동코드"] = resident["행정기관코드"].str.strip().str.zfill(10)

if not living["기준년월"].astype(str).eq("202609").all():
    raise ValueError("생활인구 파일에 2026년 9월 외 자료가 포함되어 있습니다.")
resident = resident.loc[
    resident["기준연월"].astype(str).eq("2026-09-30")
    & resident["시도명"].eq("부산광역시")
].copy()

pop_cols = ["주거인구_일평균", "직장인구_일평균", "방문인구_일평균"]
for col in pop_cols:
    living[col] = pd.to_numeric(living[col], errors="coerce")
living["시간대_생활인구"] = living[pop_cols].sum(axis=1, min_count=3)
records_per_dong = living.groupby("행정동코드").size()
if not records_per_dong.eq(24).all():
    raise ValueError("행정동별 시간대 자료가 24개씩 있는지 확인하세요.")

# 시간대별 값을 합산하지 않고 평균해 하루 인구를 중복 집계하지 않습니다.
living_dong = (
    living.groupby("행정동코드", as_index=False)["시간대_생활인구"]
    .mean()
    .rename(columns={"시간대_생활인구": "생활인구_24시간평균"})
)
resident["주민등록인구"] = pd.to_numeric(resident["전체 당월인구수"], errors="coerce")
resident_dong = resident[["행정동코드", "주민등록인구"]].copy()

boundary = gpd.read_file(data_dir / "busan_hjd_20250101.geojson")
if boundary.crs is None:
    boundary = boundary.set_crs("EPSG:4326")
boundary = boundary.to_crs("EPSG:4326")
boundary["행정동코드"] = (
    boundary["adm_cd2"].astype(str).str.replace(r"\.0$", "", regex=True).str.zfill(10)
)
boundary["행정동명"] = boundary["adm_nm"]
map_data = boundary.merge(living_dong, on="행정동코드", how="left").merge(
    resident_dong, on="행정동코드", how="left"
)
if len(map_data) != 206 or map_data[
    ["생활인구_24시간평균", "주민등록인구"]
].isna().any().any():
    raise ValueError("생활인구·주민등록인구·경계 코드 결합 결과를 확인하세요.")

map_data["생활인구_주민등록인구비"] = (
    map_data["생활인구_24시간평균"] / map_data["주민등록인구"]
)
labels = ["0.50 미만", "0.50–0.79", "0.80–0.99", "1.00–1.49", "1.50 이상"]
bins = [-np.inf, 0.5, 0.8, 1.0, 1.5, np.inf]
map_data["비율_범주"] = pd.cut(
    map_data["생활인구_주민등록인구비"],
    bins=bins,
    labels=labels,
    right=False,
).astype(str)
colors = {
    "0.50 미만": "#b6d2a6",
    "0.50–0.79": "#d0df9f",
    "0.80–0.99": "#e6d681",
    "1.00–1.49": "#efa778",
    "1.50 이상": "#cb8171",
}

center_5179 = map_data.to_crs(epsg=5179).geometry.unary_union.centroid
center_4326 = gpd.GeoSeries([center_5179], crs=5179).to_crs(epsg=4326).iloc[0]
m = folium.Map(
    location=[center_4326.y, center_4326.x],
    zoom_start=10,
    tiles=None,
    control_scale=True,
)
folium.TileLayer(
    tiles="https://{s}.basemaps.cartocdn.com/light_all/{z}/{x}/{y}{r}.png",
    attr="© OpenStreetMap contributors © CARTO",
    name="CARTO Positron",
    max_zoom=19,
    control=False,
).add_to(m)

def style_feature(feature):
    category = feature["properties"].get("비율_범주")
    return {
        "fillColor": colors.get(category, "#bdbdbd"),
        "color": "#555555",
        "weight": 0.35,
        "fillOpacity": 0.68,
    }

folium.GeoJson(
    data=map_data.to_json(),
    name="생활인구 / 주민등록인구 비율",
    style_function=style_feature,
    tooltip=folium.GeoJsonTooltip(
        fields=[
            "행정동명",
            "생활인구_24시간평균",
            "주민등록인구",
            "생활인구_주민등록인구비",
            "비율_범주",
        ],
        aliases=[
            "행정동",
            "생활인구(24시간 평균)",
            "주민등록인구(9월 말)",
            "비율",
            "범주",
        ],
        localize=True,
        sticky=True,
    ),
).add_to(m)

legend_rows = "".join(
    f'<div style="display:flex;align-items:center;gap:7px;margin:3px 0;">'
    f'<span style="width:15px;height:13px;background:{colors[label]};border:1px solid #999"></span>{label}</div>'
    for label in labels
)
legend_html = (
    '<div style="position:fixed;z-index:9999;right:22px;bottom:28px;background:white;'
    'padding:10px 13px;border-radius:6px;box-shadow:0 1px 6px #0003;font-size:12px;">'
    '<b>생활인구 / 주민등록인구</b>' + legend_rows + '</div>'
)
m.get_root().html.add_child(folium.Element(legend_html))
folium.LayerControl(collapsed=False).add_to(m)

# 노트북 셀 안에서 바로 확인합니다. 통합 대시보드 HTML 파일은 덮어쓰지 않습니다.
display(m)


## 인구밀집구역(DID) 격자 변화 지도

부산의 500m 격자를 투명 회색으로 깔고, DID 상태가 유지·탈락·신규인 격자만 색상으로 구분합니다. 툴팁에는 격자 코드 없이 2015년과 2024년 인구를 표시합니다.

In [ ]:
import json

# 분석 결과와 행정동 경계를 불러옵니다.
boundary = gpd.read_file(data_dir / "busan_hjd_20250101.geojson").to_crs("EPSG:4326")
boundary_json = json.loads(boundary.to_json())

# 분석 결과 GeoJSON을 불러옵니다.
derived_dir = data_dir / "derived"
with open(derived_dir / "busan_500m_grid.geojson", encoding="utf-8") as f:
    all_grid = json.load(f)
with open(derived_dir / "did_change_500m.geojson", encoding="utf-8") as f:
    did_change = json.load(f)

all_gdf = gpd.GeoDataFrame.from_features(all_grid["features"], crs="EPSG:4326")
minx, miny, maxx, maxy = all_gdf.total_bounds
did_map = folium.Map(
    location=[(miny + maxy) / 2, (minx + maxx) / 2], zoom_start=10,
    tiles="https://{s}.basemaps.cartocdn.com/light_all/{z}/{x}/{y}{r}.png",
    attr="© OpenStreetMap contributors © CARTO", control_scale=True,
)
did_map.fit_bounds([[miny, minx], [maxy, maxx]])

# 모든 격자는 투명 회색으로 표시합니다. 범례는 만들지 않습니다.
folium.GeoJson(
    all_grid, name="부산 500m 격자",
    style_function=lambda feature: {"color":"#ffffff", "weight":0.15, "fillColor":"#8a928d", "fillOpacity":0.20},
    control=False,
).add_to(did_map)
status_colors = {"유지":"#8fbf9e", "탈락":"#cb8171", "신규":"#6b9ec8"}
folium.GeoJson(
    did_change, name="DID 격자 상태",
    style_function=lambda feature: {
        "color":"#ffffff", "weight":0.25,
        "fillColor":status_colors.get(feature["properties"].get("status"), "#8a928d"), "fillOpacity":0.82,
    },
    tooltip=folium.GeoJsonTooltip(
        fields=["status", "population_2015", "population_2024"],
        aliases=["상태", "2015년 인구", "2024년 인구"], localize=True, sticky=False,
    ),
).add_to(did_map)
folium.GeoJson(
    boundary_json, name="행정동 경계",
    style_function=lambda feature: {"color":"#40564c", "weight":0.65, "fillOpacity":0},
    control=False,
).add_to(did_map)
did_map.get_root().html.add_child(folium.Element(
    '<div style="position:fixed;bottom:28px;left:12px;z-index:9999;background:#fff;padding:8px 10px;border-radius:7px;box-shadow:0 1px 5px #777;font-size:12px">'
    '<b>DID 격자 상태</b><br><span style="color:#8fbf9e">■</span> 유지 &nbsp; '
    '<span style="color:#cb8171">■</span> 탈락 &nbsp; <span style="color:#6b9ec8">■</span> 신규</div>'
))
display(did_map)


## 행정동별 DID 변화 지도와 구·군별 집계표

행정동별 유지·탈락·신규 격자 수를 지도 툴팁으로 확인하고, 표는 구·군 단위 합계만 표시합니다.

In [ ]:
with open(derived_dir / "busan_did_change_by_dong.geojson", encoding="utf-8") as f:
    dong_data = json.load(f)
dong_gdf = gpd.GeoDataFrame.from_features(dong_data["features"], crs="EPSG:4326")
minx, miny, maxx, maxy = dong_gdf.total_bounds
route_map = folium.Map(
    location=[(miny + maxy) / 2, (minx + maxx) / 2], zoom_start=10,
    tiles="https://{s}.basemaps.cartocdn.com/light_all/{z}/{x}/{y}{r}.png",
    attr="© OpenStreetMap contributors © CARTO", control_scale=True,
)
route_map.fit_bounds([[miny, minx], [maxy, maxx]])
route_colors = {
    "유지":"#b6d2a6", "유지·일부탈락":"#e6d681", "유지·신규포함":"#d0df9f",
    "탈락":"#cb8171", "신규":"#6b9ec8",
}
folium.GeoJson(
    dong_data, name="행정동 DID 변화 유형",
    style_function=lambda feature: {
        "color":"#ffffff", "weight":0.45,
        "fillColor":route_colors.get(feature["properties"].get("route"), "#d0d3d1"), "fillOpacity":0.72,
    },
    tooltip=folium.GeoJsonTooltip(
        fields=["adm_nm", "route", "did2015", "did2024", "kept", "lost", "gained"],
        aliases=["행정동", "변화 유형", "2015 DID 격자", "2024 DID 격자", "유지", "탈락", "신규"],
        localize=True, sticky=False,
    ),
).add_to(route_map)
display(route_map)

# 행정동 이름의 두 번째 항목(구·군)을 기준으로 집계합니다.
dong_gdf["구·군"] = dong_gdf["adm_nm"].str.split().str[1]
gu_summary = (
    dong_gdf.groupby("구·군", as_index=False)[["did2015", "did2024", "kept", "lost", "gained"]]
    .sum()
    .rename(columns={"did2015":"2015 DID 격자", "did2024":"2024 DID 격자", "kept":"유지", "lost":"탈락", "gained":"신규"})
    .sort_values("구·군")
)
total = {column: gu_summary[column].sum() for column in gu_summary.columns if column != "구·군"}
total["구·군"] = "부산 전체"
gu_summary = pd.concat([gu_summary, pd.DataFrame([total])], ignore_index=True)
display(gu_summary.style.format({c:"{:,.0f}" for c in gu_summary.columns if c != "구·군"}).set_properties(**{"text-align":"right"}).set_table_styles([{"selector":"th", "props":[("text-align","right"), ("background-color","#eef3ef")]}]))
